# Strategic Voting Prediction

| Block | Abschnitt der Arbeit | Ausgabe |
|---|---|---|
| 1 Introduction | 5.1.2, Anhang C.5 | Tabelle 6, Parameter, Effekt-Matrizen, Best-Response-Abbildung, PSE und Krümmungsmatrizen |
| 2 FTL | 5.2 und 5.3.2 | Tabelle 3 und die Werte im Text zu Tabelle 3 |
| 3 Equality | 5.4.2 | Tabelle 5 und die Werte im Text zu Tabelle 5 |

Die Tabellen werden wieder in Tables geschrieben. Alle übrigen Werte erscheinen als Ausgabe unter den Zellen. 

### Imports und Ausgabeordner
Lädt die benötigten Pakete und legt den Ordner `Tables` an. `write_table` schreibt die Kopf- und Datenzeilen einer Tabelle in diesen Ordner.

In [1]:
import os
import itertools
import numpy as np, pandas as pd
from numpy import linalg as la
from scipy.sparse import random as scirand

tab_dir = '../Tables'                                     # Ordner fuer die LaTeX-Tabellen der Arbeit
os.makedirs(tab_dir, exist_ok=True)

def write_table(fname, lines):
    # schreibt Kopf- und Datenzeilen einer Tabelle der Arbeit (ohne LaTeX-Formatierung) nach Tables/fname
    path = os.path.join(tab_dir, fname)
    with open(path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print('Tabelle geschrieben:', os.path.normpath(path))

## 1 Introduction (Abschnitt 5.1.2)
Spiel, Parameter, Effekt-Matrizen, Best-Response-Abbildung, PSE und Krümmungsmatrizen des Voting-Spiels.

### Spiel aufsetzen
Legt die Parameter fest (Batchgröße `s6_m`, Dimension `s6_d`, Box von `s6_lo` bis `s6_hi`, Rauschen `s6_muw` und `s6_sw`, Inputverteilung `s6_muth` und `s6_sth`, Basismodell `s6_B`, Regularisierung `s6_lam`) und zieht die vier Effekt-Matrizen mit Seed 2. Definiert den erwarteten Verlust, die Best Response und die Lücken einer Mischung (`s6_eps`) und berechnet den PSE `s6_xstar` mit `s6_nit` = 3000 Best-Response-Iterationen ab dem Nullprofil. Alle weiteren Zellen bauen auf dieser Zelle auf. Keine Ausgabe.

In [2]:
s6_m, s6_d = 10, 2
s6_lam = np.array([1.0, 1.0]); s6_lo, s6_hi = -100.0, 100.0
s6_muw = np.array([0.0, 0.0]); s6_sw = 0.01
s6_muth = 1.0; s6_sth = 1.0; s6_B = np.array([3.0, 3.0]); dens = 0.5
np.random.seed(2)
s6_A1  = scirand(s6_m, s6_d, density=dens).toarray()
s6_Ac1 = 0.5 * scirand(s6_m, s6_d, density=dens).toarray()
s6_A2  = scirand(s6_m, s6_d, density=dens).toarray()
s6_Ac2 = 0.5 * scirand(s6_m, s6_d, density=dens).toarray()
s6_Aown = [s6_A1, s6_A2]; s6_Acom = [s6_Ac1, s6_Ac2]
s6_ones = np.ones(s6_m)
s6_Mth = s6_muth * np.ones((s6_d, s6_m))                                   # E[Theta]
s6_Sig = s6_m * s6_muth**2 * np.ones((s6_d, s6_d)) + s6_m * s6_sth**2 * np.eye(s6_d)  # E[Theta Theta^T]

def s6_ci(x, i):                                # induced mean-demand coupling c_i(x)
    return s6_Aown[i] @ x[i] + s6_Acom[i] @ x[1 - i]
def s6_EL(i, xip, xdata):                       # E[1/2||z_i(xdata)-Th^T xip||^2] + lam/2||xip||^2
    mr = s6_muw[i] * s6_ones + s6_ci(xdata, i) + s6_Mth.T @ (s6_B - xip)
    return 0.5 * (mr @ mr + s6_m * s6_sw**2 + s6_m * s6_sth**2 * ((s6_B - xip) @ (s6_B - xip))) \
           + 0.5 * s6_lam[i] * (xip @ xip)
def s6_Ldep(x):
    return np.array([s6_EL(i, x[i], x) for i in range(2)])
def s6_BR(i, cbar):                             # best response to data with mean coupling cbar
    return la.solve(s6_Sig + s6_lam[i] * np.eye(s6_d), s6_Mth @ (s6_muw[i] * s6_ones + cbar) + s6_Sig @ s6_B)
def s6_G(x):                                    # one-step retraining map
    return np.clip(np.vstack([s6_BR(i, s6_ci(x, i)) for i in range(2)]), s6_lo, s6_hi)
def s6_eps(X):
    X = np.asarray(X); M = len(X)
    dep = np.mean([s6_Ldep(x) for x in X], axis=0)
    cbar = [np.mean([s6_ci(x, i) for x in X], axis=0) for i in range(2)]
    star = [np.clip(s6_BR(i, cbar[i]), s6_lo, s6_hi) for i in range(2)]
    mn = np.array([np.mean([s6_EL(i, star[i], x) for x in X]) for i in range(2)])
    gap = dep - mn
    return dict(gap=gap, eps=gap.max(), eps_sum=gap.sum(), block=int(np.argmax(gap)), M=M)

# spectral radius of the retraining Jacobian + PSE (iterate the contraction)
s6_Jrows = []
for i in range(2):
    Si = la.inv(s6_Sig + s6_lam[i] * np.eye(s6_d))
    s6_Jrows.append([Si @ (s6_Mth @ (s6_Aown[i] if j == i else s6_Acom[i])) for j in range(2)])
s6_J = np.block(s6_Jrows)
s6_rho = np.abs(la.eigvals(s6_J)).max()
s6_xstar = np.zeros((2, s6_d)); s6_nit = 3000                     # Anzahl der Best-Response-Iterationen
for _ in range(s6_nit): s6_xstar = s6_G(s6_xstar)

### Tabelle 6: Effekt-Matrizen (Anhang C.5)
Gibt die vier Effekt-Matrizen aus und schreibt Kopf- und Datenzeilen nach `Tables/Table6_voting_P.tex`. Einträge, die genau 0 sind, stehen dort als 0.

In [3]:
s6_tab = pd.DataFrame(np.hstack([s6_A1, s6_Ac1, s6_A2, s6_Ac2]), index=range(1, s6_m + 1),
                      columns=pd.MultiIndex.from_product([['P_1 (s6_A1)', 'P^c_1 (s6_Ac1)', 'P_2 (s6_A2)', 'P^c_2 (s6_Ac2)'], [1, 2]]))
print(s6_tab.round(3).to_string())
print()
write_table('Table6_voting_P.tex',
            [r"$j$ & \multicolumn{2}{c|}{$\cPerf_1$} & \multicolumn{2}{c|}{$\cPerf_1^c$} & \multicolumn{2}{c|}{$\cPerf_2$} & \multicolumn{2}{c|}{$\cPerf_2^c$} \\"]
            + ["%d & %s \\\\" % (j, " & ".join("0" if v == 0 else "%.3f" % v for v in row))
               for j, row in zip(s6_tab.index, s6_tab.values)])

   P_1 (s6_A1)        P^c_1 (s6_Ac1)        P_2 (s6_A2)        P^c_2 (s6_Ac2)       
             1      2              1      2           1      2              1      2
1        0.184  0.847          0.482  0.000       0.293  0.994          0.252  0.000
2        0.080  0.000          0.000  0.250       0.357  0.800          0.000  0.000
3        0.000  0.529          0.000  0.000       0.000  0.971          0.457  0.413
4        0.494  0.000          0.350  0.290       0.000  0.602          0.191  0.206
5        0.135  0.000          0.000  0.000       0.524  0.000          0.000  0.384
6        0.854  0.000          0.171  0.081       0.000  0.000          0.000  0.182
7        0.000  0.000          0.000  0.000       0.000  0.765          0.000  0.000
8        0.000  0.505          0.000  0.284       0.067  0.169          0.000  0.171
9        0.000  0.514          0.214  0.000       0.000  0.000          0.000  0.439
10       0.785  0.000          0.445  0.397       0.000  0.000   

### Werte für Abschnitt 5.1.2
Parameter des Spiels, Form und Wertebereich der Effekt-Matrizen, zweite Momente von Input und Rauschen (Assumption 3), die Matrix der zweiten Momente des Inputs mit den Eigenwerten des quadratischen Koeffizienten, der konstante Teil j der Best-Response-Abbildung (mit Prüfung, ob er der ersten Best Response ab dem Nullprofil und der Best Response ohne performativen Effekt entspricht), Eigenwerte, Spektralradius und Operatornorm der Jacobi-Matrix, der PSE, der Fehler der Best-Response-Iteration ab dem Nullprofil und der Ecken-Check, ob die Projektion jemals aktiv wird.

In [4]:
# Parameter des Spiels
print("Theta_i = [%g, %g]^%d,  k = %d,  mu_x = %g,  sigma_x = %g,  sigma_w = %g,  N = %s,  tau = %s,  density = %g"
      % (s6_lo, s6_hi, s6_d, s6_m, s6_muth, s6_sth, s6_sw, s6_B, s6_lam, dens))

# Effekt-Matrizen: Form, Anzahl der Eintraege ungleich 0, Wertebereich und Vorzeichen
print("\nEffekt-Matrizen (Form %d x %d):" % s6_A1.shape)
for name, M in [('P_1   (s6_A1) ', s6_A1), ('P_2   (s6_A2) ', s6_A2), ('P^c_1 (s6_Ac1)', s6_Ac1), ('P^c_2 (s6_Ac2)', s6_Ac2)]:
    nz = M[M != 0]
    print("  %s %2d/%d Eintraege ungleich 0, von %.3f bis %.3f,  alle >= 0: %s"
          % (name, nz.size, M.size, nz.min(), nz.max(), bool((M >= 0).all())))

# zweite Momente von Input und Rauschen (Assumption 3)
print("\nErwartete quadrierte Frobeniusnorm eines Input-Batches (Spur von Sigma_x): %g,   erwartete quadrierte Norm des Rauschens: %s"
      % (np.trace(s6_Sig), [float(s6_m * (s6_sw**2 + s6_muw[i]**2)) for i in range(2)]))
print("M_x = E[x] (Form %d x %d), alle Eintraege = %s" % (*s6_Mth.shape, np.unique(s6_Mth)))
print("Sigma_x = E[x x^T] =\n", s6_Sig)
for i in range(2):
    print("Eigenwerte von A_%d = Sigma_x + tau_%d I: %s"
          % (i + 1, i + 1, np.round(la.eigvalsh(s6_Sig + s6_lam[i] * np.eye(s6_d)), 4)))

# Best-Response-Abbildung ohne Projektion: G(theta) = c + J theta
t41_c    = np.concatenate([la.solve(s6_Sig + s6_lam[i] * np.eye(s6_d), s6_Sig @ s6_B) for i in range(2)])
t41_xsol = la.solve(np.eye(2 * s6_d) - s6_J, t41_c).reshape(2, s6_d)   # (I-J) theta* = c
print("\nj (t41_c) je Spieler:", np.round(t41_c.reshape(2, s6_d), 3).tolist())
print("j gleich der ersten Best Response ab dem Nullprofil: %s;  gleich der Best Response ohne performativen Effekt: %s"
      % (bool(np.allclose(s6_G(np.zeros((2, s6_d))), t41_c.reshape(2, s6_d))),
         bool(np.allclose(np.vstack([s6_BR(i, np.zeros(s6_m)) for i in range(2)]), t41_c.reshape(2, s6_d)))))
print("Eigenwerte von J: %s   ->   rho(J) = %.2f,   Operatornorm ||J||_2 = %.2f"
      % (np.round(la.eigvals(s6_J).real, 3) + 0.0, s6_rho, la.norm(s6_J, 2)))
print("theta* = (I - J)^(-1) j:", np.round(t41_xsol, 3).tolist())
print("s6_xstar aus %d Best-Response-Iterationen: Abweichung zu theta* %.2e,  Abstand ||G(s6_xstar) - s6_xstar|| = %.2e"
      % (s6_nit, la.norm(t41_xsol - s6_xstar), la.norm(s6_G(s6_xstar) - s6_xstar)))

# Best-Response-Iteration ab dem Nullprofil: Fehler zum PSE nach jeder Iteration
t41_x = np.zeros((2, s6_d)); t41_err = []
for _ in range(40):
    t41_x = s6_G(t41_x); t41_err.append(la.norm(t41_x - s6_xstar))
print("Best-Response-Iteration: Fehler nach 10 Iterationen %.1e, ab Iteration %d bis Iteration %d durchgehend unter 1e-15"
      % (t41_err[9], next(k + 1 for k in range(len(t41_err)) if max(t41_err[k:]) < 1e-15), len(t41_err)))

# Ecken-Check: u(theta') ist affin in theta', also liegt das Maximum von |u(theta')_j| in einer Ecke von Theta
def t41_u(x):                                  # unbeschraenkte Minimierer beider Spieler (ohne Clipping)
    return np.vstack([s6_BR(i, s6_ci(x, i)) for i in range(2)])
t41_corners = [np.array(v).reshape(2, s6_d) for v in itertools.product([s6_lo, s6_hi], repeat=2 * s6_d)]
t41_umax = max(np.abs(t41_u(x)).max() for x in t41_corners)
print("\nEcken-Check ueber %d Ecken: max |u(theta')_j| = %.2f, Grenze der Box %g  ->  Projektion nie aktiv: %s"
      % (len(t41_corners), t41_umax, s6_hi, bool(t41_umax < s6_hi)))

Theta_i = [-100, 100]^2,  k = 10,  mu_x = 1,  sigma_x = 1,  sigma_w = 0.01,  N = [3. 3.],  tau = [1. 1.],  density = 0.5

Effekt-Matrizen (Form 10 x 2):
  P_1   (s6_A1)  10/20 Eintraege ungleich 0, von 0.080 bis 0.854,  alle >= 0: True
  P_2   (s6_A2)  10/20 Eintraege ungleich 0, von 0.067 bis 0.994,  alle >= 0: True
  P^c_1 (s6_Ac1) 10/20 Eintraege ungleich 0, von 0.081 bis 0.482,  alle >= 0: True
  P^c_2 (s6_Ac2) 10/20 Eintraege ungleich 0, von 0.171 bis 0.457,  alle >= 0: True

Erwartete quadrierte Frobeniusnorm eines Input-Batches (Spur von Sigma_x): 40,   erwartete quadrierte Norm des Rauschens: [0.001, 0.001]
M_x = E[x] (Form 2 x 10), alle Eintraege = [1.]
Sigma_x = E[x x^T] =
 [[20. 10.]
 [10. 20.]]
Eigenwerte von A_1 = Sigma_x + tau_1 I: [11. 31.]
Eigenwerte von A_2 = Sigma_x + tau_2 I: [11. 31.]

j (t41_c) je Spieler: [[2.903, 2.903], [2.903, 2.903]]
j gleich der ersten Best Response ab dem Nullprofil: True;  gleich der Best Response ohne performativen Effekt: True
Eigenwerte 

### Krümmungsmatrizen und Assumption 5
Stellt die Krümmungsmatrizen der beiden Spieler (`s8_Hi`) und die gemeinsame Krümmungsmatrix (`s8_H`) auf und gibt ihre Eigenwerte aus. Definiert außerdem `s8_parts`, das eine Mischung in Zentrum und Streuung zerlegt und daraus alle Spalten von Tabelle 5 berechnet.

In [5]:
# --- Kruemmungsmatrizen H_i in geschlossener Form, Reihenfolge theta = (theta_1, theta_2) ---
# H_1, H_2 sind 2x2-Blockmatrizen mit 2x2-Bloecken, also 4x4
s8_n = 2 * s6_d; s8_I = np.eye(s6_d); s8_O = np.zeros((s6_d, s6_d))
s8_sym = lambda M: 0.5 * (M + M.T)

def s8_Hmats():
    # H_1, H_2 aus den aktuellen Effekt-Matrizen (im Code: Mth = M_x, Sig = Sigma_x, lam = omega)
    MA  = [s6_Mth @ s6_Aown[i] for i in range(2)]                        # M_x A_i
    MAc = [s6_Mth @ s6_Acom[i] for i in range(2)]                        # M_x A^c_i
    return [np.block([[0.5*(s6_Sig + s6_lam[0]*s8_I) - s8_sym(MA[0]), -0.5*MAc[0]], [-0.5*MAc[0].T, s8_O]]),
            np.block([[s8_O, -0.5*MAc[1].T], [-0.5*MAc[1], 0.5*(s6_Sig + s6_lam[1]*s8_I) - s8_sym(MA[1])]])]

s8_Hi = s8_Hmats(); s8_H = s8_Hi[0] + s8_Hi[1]
s8_ev, s8_V = la.eigh(s8_H); s8_lmin, s8_lmax = s8_ev[0], s8_ev[-1]

def s8_parts(X, Hi=None, lmin=None):
    # Zerlegung von Lemma 1 fuer die uniforme Mischung ueber die Profile X = (M, 2, d)
    Hi = s8_Hi if Hi is None else Hi; lmin = s8_lmin if lmin is None else lmin
    X = np.asarray(X, float); Y = X.reshape(len(X), -1)
    c = Y.mean(0); S = (Y - c).T @ (Y - c) / len(Y)            # Zentrum theta^c und Kovarianzmatrix Sigma
    r = s6_eps(X); rc = s6_eps(c.reshape(1, 2, s6_d))          # eps(phi) und eps(theta^c) direkt aus Definition 9
    HS = np.array([np.sum(h * S) for h in Hi])                 # <H_i, Sigma>
    return dict(M=len(X), tr=np.trace(S), eps=r['eps_sum'], eps_c=rc['eps_sum'], HS=HS.sum(),
                delta=r['eps_sum'] - rc['eps_sum'] - HS.sum(), bound=lmin * np.trace(S),
                diff=HS.sum() - lmin * np.trace(S), gap=r['gap'], delta_i=r['gap'] - rc['gap'] - HS)

# Assumption 5: H positiv definit, die H_i allein dagegen indefinit
for i in range(2):
    w = la.eigvalsh(s8_Hi[i])
    print("H_%d allein: lambda_min = %.2f, Eigenwerte %s  ->  indefinit: %s"
          % (i + 1, w[0], np.round(w, 4) + 0.0, bool(w[0] < 0 < w[-1])))
print("H = H_1 + H_2 (%dx%d): Eigenwerte %s  ->  positiv definit: %s"
      % (s8_n, s8_n, np.round(s8_ev, 2), bool(s8_lmin > 0)))

H_1 allein: lambda_min = -0.21, Eigenwerte [-0.2067  0.      5.4991 10.7808]  ->  indefinit: True
H_2 allein: lambda_min = -0.24, Eigenwerte [-0.2399  0.      5.0632 10.6351]  ->  indefinit: True
H = H_1 + H_2 (4x4): Eigenwerte [ 4.89  5.5   7.7  13.45]  ->  positiv definit: True


## 2 FTL (Abschnitte 5.2 und 5.3.2)
100 unabhängige Läufe mit exaktem Follow-the-Leader (Seeds 100 bis 199, Start im Nullprofil, je 8000 Runden), ausgewertet nach 100, 500, 2000 und 8000 Runden. Die gepoolte Mischung aller Läufe schätzt die gemittelte Trajektorien-Mischung, das Mittel des realisierten Regrets über die Läufe schätzt den erwarteten Regret.

### FTL und Kennzahlen je Lauf
`s7_run` spielt exaktes FTL: Der Leader löst in jeder Runde ein kleines lineares Gleichungssystem aus den laufenden Summen der bisherigen Daten, und es wird gezählt, wie oft die Projektion auf die Box gegriffen hat. `s7_quad`, `s7_stats` und `s7_gap` fassen einen Lauf für jede ausgewertete Rundenzahl zusammen und berechnen daraus die Lücken der gepoolten Mischung und den realisierten Regret. Keine Ausgabe.

In [6]:
def s7_run(T, seed):
    # Exaktes FTL auf den realisierten Verlusten; zaehlt, ob die Projektion auf Theta je aktiv war
    r = np.random.default_rng(seed)
    X = np.zeros((T, 2, s6_d)); TH = np.zeros((T, s6_d, s6_m)); Z = np.zeros((T, 2, s6_m))
    G = [np.zeros((s6_d, s6_d)) for _ in range(2)]; b = [np.zeros(s6_d) for _ in range(2)]
    xt = np.zeros((2, s6_d)); nclip = 0
    for t in range(1, T + 1):
        X[t-1] = xt
        Th = r.normal(s6_muth, s6_sth, size=(s6_d, s6_m)); TH[t-1] = Th
        for i in range(2):
            Z[t-1, i] = r.normal(s6_muw[i], s6_sw, size=(s6_m,)) + s6_ci(xt, i) + Th.T @ s6_B
        nxt = np.zeros((2, s6_d))
        for i in range(2):
            G[i] += Th @ Th.T; b[i] += Th @ Z[t-1, i]
            u = la.solve(G[i] + t*s6_lam[i]*np.eye(s6_d), b[i])
            nclip += int(np.any(np.abs(u) > s6_hi)); nxt[i] = np.clip(u, s6_lo, s6_hi)
        xt = nxt
    return X, TH, Z, nclip

def s7_quad(i, a, q1, q2):
    # Mittel von s6_EL(i, a, x) ueber Profile x, gegeben q1 = Mittel von q = mu_w + c_i(x) und q2 = Mittel von ||q||^2
    v = s6_Mth.T @ (s6_B - a)
    return 0.5*(q2 + 2*q1 @ v + v @ v + s6_m*s6_sw**2 + s6_m*s6_sth**2*((s6_B - a) @ (s6_B - a))) + 0.5*s6_lam[i]*(a @ a)

def s7_stats(X, TH, Z, Ts):
    # je Praefix T: Kennzahlen fuer g_i der gepoolten Mischung und realisierter Regret_i(T)/T
    T0 = len(X); out = []
    dep = np.array([s6_Ldep(x) for x in X])                                        # R_i(theta_i^t; theta^t)
    q = np.stack([s6_muw[i] + np.stack([s6_ci(x, i) for x in X]) for i in range(2)], 1)  # (T,2,k)
    qq = np.einsum('tik,tik->ti', q, q)
    res = Z - np.einsum('tdk,tid->tik', TH, X)
    real = 0.5*np.einsum('tik,tik->ti', res, res) + 0.5*s6_lam[None]*np.einsum('tid,tid->ti', X, X)
    GG = np.cumsum(np.einsum('tdk,tek->tde', TH, TH), 0)                           # laufende Summen
    bb = np.cumsum(np.einsum('tdk,tik->tid', TH, Z), 0); zz = np.cumsum(np.einsum('tik,tik->ti', Z, Z), 0)
    for T in Ts:
        reg = np.zeros(2); unclip = True
        for i in range(2):
            u = la.solve(GG[T-1] + T*s6_lam[i]*np.eye(s6_d), bb[T-1, i]); unclip &= bool(np.all(np.abs(u) < s6_hi))
            a = np.clip(u, s6_lo, s6_hi)                                           # bester fester Parameter im Nachhinein
            best = 0.5*zz[T-1, i] - a @ bb[T-1, i] + 0.5*a @ GG[T-1] @ a + 0.5*T*s6_lam[i]*(a @ a)
            reg[i] = (real[:T, i].sum() - best)/T
        out.append(dict(dep=dep[:T].mean(0), q1=q[:T].mean(0), q2=qq[:T].mean(0), reg=reg, unclip=unclip))
    return out

def s7_gap(dep, q1, q2):
    # g_i der gepoolten Mischung aus den gemittelten Kennzahlen (Definition 15)
    g = np.zeros(2)
    for i in range(2):
        star = np.clip(s6_BR(i, q1[i] - s6_muw[i]), s6_lo, s6_hi)
        g[i] = dep[i] - s7_quad(i, star, q1[i], q2[i])
    return g

### 100 FTL-Läufe
Führt die Läufe aus und speichert je Lauf und Rundenzahl die Kennzahlen aus `s7_stats`. Ausgegeben wird, wie oft die Projektion im Leader aktiv war (Abschnitt 5.2) und ob der beste feste Parameter im Nachhinein je geclippt wurde, dazu eine Kontrolle der Lückenberechnung gegen `s6_eps` und die Prüfung, ob jeder Lauf im Nullprofil beginnt. Dauert etwa 1 Minute.

In [7]:
# --- 100 unabhaengige Laeufe, je 8000 Runden exaktes FTL ab theta^1 = 0 ---
s7_n, s7_Tmax, s7_seed0 = 100, 8000, 100
s7_Ts = [100, 500, 2000, 8000]; s7_K = len(s7_Ts)
s7_dep = np.zeros((s7_n, s7_K, 2)); s7_q1 = np.zeros((s7_n, s7_K, 2, s6_m)); s7_q2 = np.zeros((s7_n, s7_K, 2))
s7_reg = np.zeros((s7_n, s7_K, 2)); s7_ok = True; s7_nclip = 0
s7_runs = []
s7_x0 = True                                                     # beginnt jeder Lauf im Nullprofil?
print("%d Laeufe (Seeds %d-%d) mit je %d Runden, Auswertung bei T = %s"
      % (s7_n, s7_seed0, s7_seed0 + s7_n - 1, s7_Tmax, s7_Ts))
for s in range(s7_n):
    X, TH, Z, nc = s7_run(s7_Tmax, seed=s7_seed0 + s); s7_nclip += nc
    s7_x0 &= bool(np.all(X[0] == 0))
    for k, r in enumerate(s7_stats(X, TH, Z, s7_Ts)):
        s7_dep[s, k], s7_q1[s, k], s7_q2[s, k], s7_reg[s, k] = r['dep'], r['q1'], r['q2'], r['reg']
        s7_ok &= r['unclip']
    if s < 5: s7_runs.append(X[:100])
    if (s + 1) % 25 == 0:
        print("  %d/%d Laeufe fertig" % (s + 1, s7_n))

s7_chk = s6_eps(np.concatenate(s7_runs))['gap'] - s7_gap(s7_dep[:5, 0].mean(0), s7_q1[:5, 0].mean(0), s7_q2[:5, 0].mean(0))
print("Kontrolle gepoolte Luecke gegen s6_eps: max. Abweichung %.1e" % np.abs(s7_chk).max())
print("Projektion im FTL-Leader aktiv: %d mal;  bester fester Parameter im Nachhinein nie geclippt: %s" % (s7_nclip, s7_ok))
print("Start im Nullprofil in allen Laeufen: %s" % s7_x0)

100 Laeufe (Seeds 100-199) mit je 8000 Runden, Auswertung bei T = [100, 500, 2000, 8000]


  25/100 Laeufe fertig


  50/100 Laeufe fertig


  75/100 Laeufe fertig


  100/100 Laeufe fertig
Kontrolle gepoolte Luecke gegen s6_eps: max. Abweichung 2.8e-14
Projektion im FTL-Leader aktiv: 0 mal;  bester fester Parameter im Nachhinein nie geclippt: True
Start im Nullprofil in allen Laeufen: True


### Schätzer und gepaartes Bootstrap
Lücke der gepoolten Mischung, Toleranz aus dem mittleren Regret, ihre Differenz Delta sowie ein gepaartes Bootstrap über die Läufe mit 10 000 Ziehungen (Seed 7) für Standardfehler und 95%-Konfidenzintervalle. Keine Ausgabe.

In [8]:
def s7_est(idx):
    g = np.array([s7_gap(s7_dep[idx, k].mean(0), s7_q1[idx, k].mean(0), s7_q2[idx, k].mean(0)) for k in range(s7_K)])
    return g, s7_reg[idx].mean(0)
s7_g, s7_R = s7_est(np.arange(s7_n))
s7_G, s7_eT = s7_g.max(1), np.maximum(0, s7_R.max(1))
s7_B = 10000; s7_rng = np.random.default_rng(7)
s7_bg = np.zeros((s7_B, s7_K, 2)); s7_bR = np.zeros((s7_B, s7_K, 2))
for b in range(s7_B):
    s7_bg[b], s7_bR[b] = s7_est(s7_rng.integers(0, s7_n, s7_n))
s7_bG, s7_beT = s7_bg.max(2), np.maximum(0, s7_bR.max(2))
s7_D = s7_G - s7_eT; s7_bD = s7_bG - s7_beT
s7_lo95, s7_hi95 = np.percentile(s7_bD, [2.5, 97.5], axis=0)

### Tabelle 3
Gibt die Tabelle lesbar aus (in Klammern die Bootstrap-Standardfehler) und schreibt Kopf- und Datenzeilen nach `Tables/Table3_FTL_VS.tex`.

In [9]:
print("Strategic Voting: %d Laeufe (Seeds %d-%d), in Klammern Bootstrap-Standardfehler (%d Ziehungen)\n"
      % (s7_n, s7_seed0, s7_seed0 + s7_n - 1, s7_B))
print("  %6s | %22s | %22s | %10s | %24s" % ("T", "G(phibar_P)", "eps_T", "Delta", "95%-KI fuer Delta"))
for k, Tc in enumerate(s7_Ts):
    print("  %6d | %.5f (%.5f) | %.5f (%.5f) | %+.5f | [%+.5f, %+.5f]"
          % (Tc, s7_G[k], s7_bG[:, k].std(ddof=1), s7_eT[k], s7_beT[:, k].std(ddof=1), s7_D[k], s7_lo95[k], s7_hi95[k]))

s7_lines = [r"$T$ & $\G(\cmixap)$ & $\hat{\ce}_T$ & $\Delta$ & 95\% CI for $\Delta$ \\"]
for k, Tc in enumerate(s7_Ts):
    s7_lines.append(r"%-4d & $%.5f \ (%.5f)$ & $%.5f \ (%.5f)$ & $%+.5f$ & $[%+.5f, %+.5f]$ \\"
                    % (Tc, s7_G[k], s7_bG[:, k].std(ddof=1), s7_eT[k], s7_beT[:, k].std(ddof=1), s7_D[k], s7_lo95[k], s7_hi95[k]))
print()
write_table('Table3_FTL_VS.tex', s7_lines)

Strategic Voting: 100 Laeufe (Seeds 100-199), in Klammern Bootstrap-Standardfehler (10000 Ziehungen)

       T |            G(phibar_P) |                  eps_T |      Delta |        95%-KI fuer Delta
     100 | 2.90282 (0.01613) | 2.85499 (0.08446) | +0.04783 | [-0.10898, +0.20229]
     500 | 0.59590 (0.00389) | 0.58727 (0.01688) | +0.00863 | [-0.02204, +0.03908]
    2000 | 0.15234 (0.00112) | 0.15033 (0.00422) | +0.00201 | [-0.00557, +0.00966]
    8000 | 0.03897 (0.00031) | 0.03846 (0.00106) | +0.00050 | [-0.00137, +0.00239]

Tabelle geschrieben: ../Tables/Table3_FTL_VS.tex


### Werte für Abschnitt 5.3.2
Streuung des realisierten Regrets über die Läufe, Auflösung des Tests (obere Konfidenzgrenze relativ zur Toleranz), die Aussage je Spieler, das Wachstum von Rundenzahl mal Lücke und die Einzelläufe, in denen die Lücke der realisierten Mischung den Regret des Laufs übersteigt.

In [10]:
# Streuung des realisierten Regrets ueber die Laeufe, fuer den Spieler mit dem groessten mittleren Regret
s7_imax = int(np.argmax(s7_R[0]))
print("Spieler mit dem groessten mittleren Regret bei T = %d: %d;  sein realisierter Regret: Mittel %.2f, Standardabweichung ueber die Laeufe %.2f"
      % (s7_Ts[0], s7_imax + 1, s7_reg[:, 0, s7_imax].mean(), s7_reg[:, 0, s7_imax].std(ddof=1)))

# groesste mit den Daten vertraegliche Verletzung = obere KI-Grenze von Delta
print("Obere KI-Grenze von Delta bei T = %d: %.4f (%.1f%% von eps_T)"
      % (s7_Ts[-1], s7_hi95[-1], 100 * s7_hi95[-1] / s7_eT[-1]))

print("\nje Spieler: g_i(phibar_P) - mittlerer Regret_i(T)/T mit 95%-Bootstrap-KI")
s7_in = True
for k, Tc in enumerate(s7_Ts):
    di = s7_bg[:, k] - s7_bR[:, k]; lo, hi = np.percentile(di, [2.5, 97.5], axis=0)
    s7_in &= bool(np.all((lo <= 0) & (hi >= 0)))
    print("  T = %5d:" % Tc, "  ".join("i=%d: g=%.5f R=%.5f  %+.5f [%+.5f, %+.5f]"
          % (i + 1, s7_g[k, i], s7_R[k, i], s7_g[k, i] - s7_R[k, i], lo[i], hi[i]) for i in range(2)))
print("Alle KIs je Spieler enthalten 0:", s7_in)

s7_TG = np.array(s7_Ts) * s7_G
print("\nT * G(phibar_P):", ", ".join("%.0f (T = %d)" % (v, Tc) for v, Tc in zip(s7_TG, s7_Ts)))
print("Zuwachs von T * G bei Vervierfachung von T:",
      ", ".join("%.1f (T = %d auf %d)" % (s7_TG[k] - s7_TG[k - 1], s7_Ts[k - 1], s7_Ts[k])
                for k in range(1, s7_K) if s7_Ts[k] == 4 * s7_Ts[k - 1]))

# Einzellaeufe: Luecke der realisierten Mischung gegen den Regret desselben Laufs
s7_ind =np.array([[s7_gap(s7_dep[s, k], s7_q1[s, k], s7_q2[s, k]).max() for k in range(s7_K)] for s in range(s7_n)])
s7_indR = s7_reg.max(2)
print("\nEinzellaeufe: G(phi_T(O)) > max_i Regret_i(T)/T in %s von %d Laeufen (je T)"
      % ([int((s7_ind[:, k] > s7_indR[:, k]).sum()) for k in range(s7_K)], s7_n))

Spieler mit dem groessten mittleren Regret bei T = 100: 2;  sein realisierter Regret: Mittel 2.85, Standardabweichung ueber die Laeufe 0.84
Obere KI-Grenze von Delta bei T = 8000: 0.0024 (6.2% von eps_T)

je Spieler: g_i(phibar_P) - mittlerer Regret_i(T)/T mit 95%-Bootstrap-KI
  T =   100: i=1: g=2.88363 R=2.81729  +0.06634 [-0.09033, +0.21549]  i=2: g=2.90282 R=2.85499  +0.04783 [-0.11132, +0.20173]
  T =   500: i=1: g=0.59031 R=0.57888  +0.01143 [-0.02005, +0.04144]  i=2: g=0.59590 R=0.58727  +0.00863 [-0.02226, +0.03899]
  T =  2000: i=1: g=0.15065 R=0.14800  +0.00265 [-0.00531, +0.01023]  i=2: g=0.15234 R=0.15033  +0.00201 [-0.00563, +0.00962]
  T =  8000: i=1: g=0.03848 R=0.03782  +0.00066 [-0.00136, +0.00258]  i=2: g=0.03897 R=0.03846  +0.00050 [-0.00138, +0.00238]
Alle KIs je Spieler enthalten 0: True

T * G(phibar_P): 290 (T = 100), 298 (T = 500), 305 (T = 2000), 312 (T = 8000)
Zuwachs von T * G bei Vervierfachung von T: 6.7 (T = 500 auf 2000), 7.1 (T = 2000 auf 8000)

Einzella

## 3 Equality (Abschnitt 5.4.2)
Zerlegung der aggregierten Lücke für zehn Typen von Mischungen in die Lücke des Zentrums und den Streuungsterm, zusammen mit der unteren Schranke für den Streuungsterm. Je Typ 100 Mischungen; nur die PSE-Punktmasse und das Paar entlang des kleinsten Eigenvektors sind deterministisch. Die FTL-Mischungen stammen aus denselben 100 Läufen wie in Block 2 (Seeds 100 bis 199) und werden hier neu gerechnet.

### Erzeugung der Mischungen
Paare um den PSE (Eigenvektor zum kleinsten Eigenwert sowie 100 Zufallsrichtungen, Seed [8,1]), Gauß-Wolken aus je 200 Profilen um den PSE und um ein festes Zufallsprofil (Standardabweichung 1 mit Seed [8,2], 0.01 mit Seed [8,4], festes Profil aus Seed 1), die realisierten FTL-Mischungen nach 100 und 8000 Runden und Zufallsmischungen aus 2 bis 50 gleichverteilten Profilen (Seed [8,6]). Eine Wolke mit einem Profil außerhalb der Box würde verworfen und neu gezogen. Dauert etwa 1 Minute.

In [11]:
def s8_ftl(T, seed):
    # exaktes FTL ab theta^1 = 0, dieselben Iterierten wie s7_run in Block 2
    r = np.random.default_rng(seed); X = np.zeros((T, 2, s6_d))
    G = [np.zeros((s6_d, s6_d)) for _ in range(2)]; b = [np.zeros(s6_d) for _ in range(2)]
    xt = np.zeros((2, s6_d))
    for t in range(1, T + 1):
        X[t-1] = xt
        Th = r.normal(s6_muth, s6_sth, size=(s6_d, s6_m)); Z = np.zeros((2, s6_m))
        for i in range(2):
            Z[i] = r.normal(s6_muw[i], s6_sw, size=(s6_m,)) + s6_ci(xt, i) + Th.T @ s6_B
        nxt = np.zeros((2, s6_d))
        for i in range(2):
            G[i] += Th @ Th.T; b[i] += Th @ Z[i]
            nxt[i] = np.clip(la.solve(G[i] + t*s6_lam[i]*np.eye(s6_d), b[i]), s6_lo, s6_hi)
        xt = nxt
    return X

s8_N = 100                                       # Mischungen je Typ, wie die 100 Laeufe in Block 2
s8_cseed = 1                                     # Seed fuer das feste Zufallsprofil theta~
s8_vmin = s8_V[:, 0].reshape(2, s6_d)            # Eigenvektor zu lambda_min(H), Laenge 1
# theta~: einmal gleichverteilt aus Theta gezogen (Abstand 10 zum Rand, damit alle Wolken-Profile in Theta liegen)
s8_xc = np.random.default_rng(s8_cseed).uniform(s6_lo + 10, s6_hi - 10, (2, s6_d))

def s8_unit(rng):
    v = rng.normal(size=(2, s6_d)); return v / la.norm(v)

def s8_clouds(center, s, k):
    # 100 Wolken aus je 200 Profilen ~ N(center, s^2 I), nur Wolken ganz in Theta; gleiches k = dieselben Zufallszahlen, nur anderes Zentrum
    global s8_rej
    rng = np.random.default_rng([8, k]); out = []
    while len(out) < s8_N:
        Z = rng.normal(0, s, (200, 2, s6_d))
        X = center + Z - Z.mean(0)                             # exakt zentrieren: theta^c = center, Sigma bleibt gleich
        if X.min() < s6_lo or X.max() > s6_hi:                 # Profil ausserhalb von Theta: Wolke verwerfen, neu ziehen
            s8_rej += 1; continue
        out.append(s8_parts(X))
    return out
s8_rej = 0                                      # Anzahl verworfener Wolken

s8_mix = {}                                      # Typ der Mischung -> Zerlegungen (s8_parts) aller Mischungen
s8_mix['theta* (PSE)']    = [s8_parts(s6_xstar[None])]                                     # deterministisch
s8_mix['theta* +- v_min'] = [s8_parts(np.stack([s6_xstar + s8_vmin, s6_xstar - s8_vmin]))]  # deterministisch
rng = np.random.default_rng([8, 1])
s8_mix['theta* +- v'] = [s8_parts(np.stack([s6_xstar + v, s6_xstar - v]))
                         for v in (s8_unit(rng) for _ in range(s8_N))]
s8_mix['Wolke um theta*, s = 1']    = s8_clouds(s6_xstar, 1.0, 2)
s8_mix['Wolke um theta*, s = 0.01'] = s8_clouds(s6_xstar, 0.01, 4)
s8_mix['Wolke um theta~, s = 1']    = s8_clouds(s8_xc, 1.0, 2)      # dieselben Wolken, um theta~ verschoben
s8_mix['Wolke um theta~, s = 0.01'] = s8_clouds(s8_xc, 0.01, 4)
s8_mix['FTL, T = 100'] = []; s8_mix['FTL, T = 8000'] = []
for s in range(s8_N):                            # dieselben 100 Laeufe wie in Block 2 (Seeds 100-199)
    X = s8_ftl(8000, seed=100 + s)
    s8_mix['FTL, T = 100'].append(s8_parts(X[:100])); s8_mix['FTL, T = 8000'].append(s8_parts(X))
    if s == 0: s8_X100, s8_X100_seed = X, 100 + s     # erster Lauf, fuer den Anteil der ersten Iterationen in Abschnitt 5.4.2
    if (s + 1) % 25 == 0:
        print("  FTL: %d/%d Laeufe fertig" % (s + 1, s8_N))
rng = np.random.default_rng([8, 6])
s8_mix['Zufallsmischung'] = [s8_parts(rng.uniform(s6_lo, s6_hi, (int(rng.integers(2, 51)), 2, s6_d)))
                             for _ in range(s8_N)]
print("Verworfene Wolken: %d" % s8_rej)

  FTL: 25/100 Laeufe fertig


  FTL: 50/100 Laeufe fertig


  FTL: 75/100 Laeufe fertig


  FTL: 100/100 Laeufe fertig
Verworfene Wolken: 0


### Tabelle 5
Mittelwert und Standardabweichung je Typ, lesbar ausgegeben (mit der Anzahl der Mischungen je Typ). Kopf- und Datenzeilen werden nach `Tables/Table5_Set_VS.tex` geschrieben, Mittelwerte mit 7 und Standardabweichungen mit 6 Ziffern. Bei Typen mit nur einer Mischung entfällt die Standardabweichung.

In [12]:
s8_cols = ['G_S(phi)', 'G_S(delta_c)', '<H,Sigma>', 'lmin*Trace']
s8_keys = ['eps', 'eps_c', 'HS', 'bound']
s8_mean = pd.DataFrame({c: [np.mean([r[k] for r in rs]) for rs in s8_mix.values()]
                        for c, k in zip(s8_cols, s8_keys)}, index=list(s8_mix))
s8_sd   = pd.DataFrame({c: [np.std([r[k] for r in rs], ddof=1) if len(rs) > 1 else 0.0 for rs in s8_mix.values()]
                        for c, k in zip(s8_cols, s8_keys)}, index=list(s8_mix))

def s8_fmt(v, digits):
    # feste Ziffernzahl, z.B. 7 Ziffern: 0.003104, 171341.9, 1773067
    if abs(v) < 0.5 * 10.0**(1 - digits): v = 0.0
    dec = max(digits - len(str(int(abs(v)))), 0)
    s = "%.*f" % (dec, v)
    if sum(ch.isdigit() for ch in s) > digits and dec > 0:     # Aufrunden hat eine Stelle ergaenzt
        s = "%.*f" % (dec - 1, v)
    return s

s8_m = {name: [s8_fmt(s8_mean.loc[name, c], 7) for c in s8_cols] for name in s8_mix}    # Mittelwert: 7 Ziffern
s8_s = {name: [s8_fmt(s8_sd.loc[name, c], 6) for c in s8_cols] for name in s8_mix}      # SD: 6 Ziffern
print("Mittelwert (SD) je Typ der Mischung\n")
print("  %-26s | %6s | " % ("Typ", "Anzahl") + " | ".join("%19s" % c for c in s8_cols))
for name in s8_mix:
    print("  %-26s | %6d | " % (name, len(s8_mix[name]))
          + " | ".join("%19s" % ("%s (%s)" % (m, s)) for m, s in zip(s8_m[name], s8_s[name])))

s8_tex = {'theta* (PSE)': r'PSE $\dirac_{\cxps}$', 'theta* +- v_min': r'$\cxps\pm v_{\min}$', 'theta* +- v': r'$\cxps\pm v$',
          'Wolke um theta*, s = 1': r'Cloud $\mathcal N(\cxps,I)$',
          'Wolke um theta*, s = 0.01': r'Cloud $\mathcal N(\cxps,0.01^2I)$',
          'Wolke um theta~, s = 1': r'Cloud $\mathcal N(\cxf,I)$',
          'Wolke um theta~, s = 0.01': r'Cloud $\mathcal N(\cxf,0.01^2I)$',
          'FTL, T = 100': r'FTL $\cmixt(\cO)$, $T=100$', 'FTL, T = 8000': r'FTL $\cmixt(\cO)$, $T=8000$',
          'Zufallsmischung': 'random mixture'}
s8_lines = [r"Type of mixture & $\Gs(\cmix)$ & $\Gs(\dirac_{\cm})$ & $\langle\cCurv,\cCov\rangle$ & $\lambda_{\min}(\cCurv)\,\mathrm{Trace}(\cCov)$ \\"]
for name in s8_mix:
    s8_row = r"%s & %s \\" % (s8_tex[name], " & ".join(s8_m[name]))
    if len(s8_mix[name]) > 1:                                    # SD nur bei Typen mit mehreren Mischungen
        s8_row += r" & %s \\" % " & ".join("(%s)" % s for s in s8_s[name])
    s8_lines.append(s8_row)
print()
write_table('Table5_Set_VS.tex', s8_lines)

Mittelwert (SD) je Typ der Mischung

  Typ                        | Anzahl |            G_S(phi) |        G_S(delta_c) |           <H,Sigma> |          lmin*Trace
  theta* (PSE)               |      1 |  0.000000 (0.00000) |  0.000000 (0.00000) |  0.000000 (0.00000) |  0.000000 (0.00000)
  theta* +- v_min            |      1 |  4.887195 (0.00000) |  0.000000 (0.00000) |  4.887195 (0.00000) |  4.887195 (0.00000)
  theta* +- v                |    100 |  7.925089 (2.06767) |  0.000000 (0.00000) |  7.925089 (2.06767) |  4.887195 (0.00000)
  Wolke um theta*, s = 1     |    100 |  31.35197 (1.81149) |  0.000000 (0.00000) |  31.35197 (1.81149) |  19.45389 (1.10405)
  Wolke um theta*, s = 0.01  |    100 |  0.003126 (0.00018) |  0.000000 (0.00000) |  0.003126 (0.00018) |  0.001938 (0.00010)
  Wolke um theta~, s = 1     |    100 |  109168.3 (1.81149) |  109137.0 (0.00000) |  31.35197 (1.81149) |  19.45389 (1.10405)
  Wolke um theta~, s = 0.01  |    100 |  109137.0 (0.00018) |  109137.0 (0.00000)

### Werte für Abschnitt 5.4.2
Residuum der Zerlegung, Prüfung der unteren Schranke samt Verhältnis zur Schranke, Prüfung auf eine weitere PSME, die Paare mit negativer Lücke für einen Spieler, die Anteile von Zentrum und Streuung bei den FTL-Mischungen, das Produkt aus Rundenzahl und Spur sowie der Anteil der ersten zehn Iterationen im Lauf mit Seed 100.

In [13]:
s8_all = [r for name, rs in s8_mix.items() if name != 'theta* (PSE)' for r in rs]          # alle Mischungen ausser der PSE
s8_oth = [r for name, rs in s8_mix.items() if name not in ('theta* (PSE)', 'theta* +- v_min') for r in rs]
print("Theorem 2, %d Mischungen ohne PSE: groesstes relatives Residuum %.1e fuer G_S und %.1e fuer die einzelnen g_i"
      % (len(s8_all), max(abs(r['delta']) / r['eps'] for r in s8_all),
         max(np.max(np.abs(r['delta_i']) / np.abs(r['gap'])) for r in s8_all)))

# Lemma 2: Schranke in allen Mischungen, Gleichheit bei der PSE und entlang des kleinsten Eigenvektors?
print("\nLemma 2 gilt fuer alle Mischungen: %s" % all(r['diff'] >= -1e-9 * max(1.0, abs(r['HS'])) for r in s8_all))
for name in ['theta* (PSE)', 'theta* +- v_min']:
    r = s8_mix[name][0]
    print("%s: <H,Sigma> = %.2f, Abstand zur Schranke %.1e  ->  Schranke mit Gleichheit erreicht: %s"
          % (name, r['HS'], r['diff'], bool(abs(r['diff']) <= 1e-9 * max(1.0, abs(r['HS'])))))
print("lambda_min(H) = %.2f" % s8_lmin)
s8_q = [r['HS'] / r['bound'] for r in s8_oth]
print("Verhaeltnis <H,Sigma> / Schranke fuer alle anderen Mischungen: %.2f bis %.2f;  lambda_max/lambda_min = %.2f  ->  alle Verhaeltnisse hoechstens so gross: %s"
      % (min(s8_q), max(s8_q), s8_lmax / s8_lmin, bool(max(s8_q) <= s8_lmax / s8_lmin * (1 + 1e-12))))

# Theorem 3: gibt es unter den getesteten Mischungen eine weitere PSME?
print("\nTheorem 3: G_S(phi) > 0 in jeder Mischung ausser der PSE: %s;  Mischungen mit g_i <= 0 fuer beide Spieler: %d von %d"
      % (all(r['eps'] > 0 for r in s8_all), sum(bool(np.all(r['gap'] <= 0)) for r in s8_all), len(s8_all)))
print("Paare theta* +- v mit negativer Luecke fuer einen Spieler: %d von %d  (in allen anderen Typen zusammen: %d)"
      % (sum(bool(np.any(r['gap'] < 0)) for r in s8_mix['theta* +- v']), len(s8_mix['theta* +- v']),
         sum(bool(np.any(r['gap'] < 0)) for name, rs in s8_mix.items() if name != 'theta* +- v' for r in rs)))

# FTL-Mischungen: Anteile von Zentrum und Streuung
print("\nFTL: Luecke des Zentrums im Mittel %.2f von G_S = %.2f (T = %d) und %.4f von G_S = %.4f (T = %d)"
      % (s8_mean.loc['FTL, T = 100', 'G_S(delta_c)'], s8_mean.loc['FTL, T = 100', 'G_S(phi)'], s8_mix['FTL, T = 100'][0]['M'],
         s8_mean.loc['FTL, T = 8000', 'G_S(delta_c)'], s8_mean.loc['FTL, T = 8000', 'G_S(phi)'], s8_mix['FTL, T = 8000'][0]['M']))
s8_sh = [100 * r['HS'] / r['eps'] for r in s8_mix['FTL, T = 8000']]
print("FTL, T = %d: Anteil der Streuung an G_S im Mittel %.1f%%" % (s8_mix['FTL, T = 8000'][0]['M'], np.mean(s8_sh)))
print("FTL: T * Trace(Sigma_T) im Mittel", ", ".join("%.1f (T = %d)" % (np.mean([r['M'] * r['tr'] for r in s8_mix[n]]), s8_mix[n][0]['M'])
                                                     for n in ['FTL, T = 100', 'FTL, T = 8000']))
s8_Y100 = s8_X100.reshape(len(s8_X100), -1)
s8_dev = np.sum((s8_Y100 - s8_Y100.mean(0))**2, 1)                   # quadrierter Abstand jeder Iteration zum Zentrum
print("Lauf mit Seed %d, T = %d: Anteil der ersten 10 Iterationen an der Spur der Kovarianzmatrix %.0f%%"
      % (s8_X100_seed, len(s8_X100), 100 * s8_dev[:10].sum() / s8_dev.sum()))

Theorem 2, 801 Mischungen ohne PSE: groesstes relatives Residuum 3.1e-11 fuer G_S und 5.8e-11 fuer die einzelnen g_i

Lemma 2 gilt fuer alle Mischungen: True
theta* (PSE): <H,Sigma> = 0.00, Abstand zur Schranke 0.0e+00  ->  Schranke mit Gleichheit erreicht: True
theta* +- v_min: <H,Sigma> = 4.89, Abstand zur Schranke 8.9e-16  ->  Schranke mit Gleichheit erreicht: True
lambda_min(H) = 4.89
Verhaeltnis <H,Sigma> / Schranke fuer alle anderen Mischungen: 1.04 bis 2.75;  lambda_max/lambda_min = 2.75  ->  alle Verhaeltnisse hoechstens so gross: True

Theorem 3: G_S(phi) > 0 in jeder Mischung ausser der PSE: True;  Mischungen mit g_i <= 0 fuer beide Spieler: 0 von 801
Paare theta* +- v mit negativer Luecke fuer einen Spieler: 6 von 100  (in allen anderen Typen zusammen: 0)

FTL: Luecke des Zentrums im Mittel 0.84 von G_S = 5.79 (T = 100) und 0.0037 von G_S = 0.0776 (T = 8000)
FTL, T = 8000: Anteil der Streuung an G_S im Mittel 95.4%
FTL: T * Trace(Sigma_T) im Mittel 68.6 (T = 100), 82.9 (T = 